In [ ]:
'''This script is used to clean the data import from sql to pivot, join and impute '''
import psycopg2
import pandas as pd

In [ ]:
from sqlalchemy import create_engine
engine = create_engine('')
# Connect to the database
master_df = pd.read_sql_query('Select * from "username".mastertable',con=engine)
#master_df

In [ ]:
interventions_df = pd.read_sql_query('SELECT * FROM "username".intervention_24_agg',con=engine)
#interventions_df

In [4]:
measurements_df = pd.read_sql_query('SELECT * FROM "lu.kuang".measurements_24_agg',con=engine)
#measurements_df 

In [5]:
prescription_df = pd.read_sql_query('SELECT * FROM "lu.kuang".prescriptions_24_agg',con=engine)
#prescription_df

In [6]:
# Select the desired columns
selected_interventions_columns = interventions_df.columns[4:13]

In [7]:
#pivoted_df = interventions_df.pivot_table(index='ADMISSION_ID', columns='ITEM_ID', values=selected_columns)
pivoted_interventions_df = interventions_df.pivot_table(index='ADMISSION_ID', columns='ITEM_ID', values=selected_interventions_columns)
#pivoted_interventions_df

In [9]:
# Flatten the column names by concatenating the label with the selected columns
new_interventions_columns = []
for column in pivoted_interventions_df.columns:
    new_interventions_column = f'{column[0]}_{column[1]}_{interventions_df.loc[interventions_df["ITEM_ID"] == column[1]]["LABEL"].values[0]}'
    new_interventions_columns.append(new_interventions_column)
#new_interventions_columns

In [11]:
# Assign the new column names
pivoted_interventions_df.columns = new_interventions_columns
# Reset index
pivoted_interventions_df.reset_index(inplace=True)
#pivoted_interventions_df

In [15]:
# Calculate the NaN percentage for each column
nan_percentage_interventions = pivoted_interventions_df.isna().mean()
#nan_percentage_interventions

In [16]:
threshold = 0.3
selected_interventions_columns = nan_percentage_interventions[nan_percentage_interventions <= threshold].index

In [17]:
# Filter the DataFrame based on selected columns
interventions_final_df = pivoted_interventions_df[selected_interventions_columns]
#interventions_final_df

In [19]:
# Select the columns from measurements
selected_measurements_columns = measurements_df.columns[4:13]
pivoted_measurements_df = measurements_df.pivot_table(index='ADMISSION_ID', columns='ITEM_ID', values=selected_measurements_columns)
#pivoted_measurements_df

In [21]:
# Flatten the column names by concatenating the label with the selected columns
new_measurements_columns = []
for column in pivoted_measurements_df.columns:
    new_measurements_column = f'{column[0]}_{column[1]}_{measurements_df.loc[measurements_df["ITEM_ID"] == column[1]]["LABEL"].values[0]}'
    new_measurements_columns.append(new_measurements_column)
#new_measurements_columns

In [23]:
# Assign the new column names
pivoted_measurements_df.columns = new_measurements_columns
# Reset index
pivoted_measurements_df.reset_index(inplace=True)

In [24]:
# Calculate the NaN percentage for each column
nan_percentage_measurements = pivoted_measurements_df.isna().mean()
#nan_percentage_measurements

In [25]:
measurements_selected_columns = nan_percentage_measurements[nan_percentage_measurements <= threshold].index

In [26]:
# Filter the DataFrame based on selected columns
measurements_final_df = pivoted_measurements_df[measurements_selected_columns]
#measurements_final_df

In [28]:
#prescription_df
selected_prescription_columns = prescription_df.columns[5:14]
pivoted_prescription_df = prescription_df.pivot_table(index='ADMISSION_ID', columns='ITEM_ID', values=selected_prescription_columns)
#pivoted_prescription_df

In [29]:
new_prescription_columns = []
for column in pivoted_prescription_df.columns:
     new_prescription_column = f'{column[0]}_{column[1]}'
     new_prescription_columns.append(new_prescription_column)
# Assign the new column names
pivoted_prescription_df.columns = new_prescription_columns
 
# Calculate the NaN percentage for each column
nan_percentage_prescription = pivoted_prescription_df.isna().mean()
nan_percentage_prescription
prescription_selected_columns = nan_percentage_prescription[nan_percentage_prescription <= threshold].index
# Filter the DataFrame based on selected columns
prescription_final_df = pivoted_prescription_df[prescription_selected_columns]
#prescription_final_df

In [37]:
measurements_value_char = pd.read_sql_query('SELECT * FROM "lu.kuang".measurements_value_char',con=engine)
measurements_value_char

,ADMISSION_ID,ITEM_ID,LABEL,VALUE_CHAR,count
0,A00000,I004,Verbal Response - Unable to Assess,Intubated or Tracheostomy,6
1,A00000,I004,Verbal Response - Unable to Assess,Sedated,6
2,A00001,I004,Verbal Response - Unable to Assess,Intubated or Tracheostomy,7
3,A00002,I004,Verbal Response - Unable to Assess,Intubated or Tracheostomy,1
4,A00002,I004,Verbal Response - Unable to Assess,Sedated,1
...,...,...,...,...,...
120639,A55684,I019,Verbal Description of Pain,Constant,3
120640,A55684,I019,Verbal Description of Pain,Throbbing,4
120641,A55685,I004,Verbal Response - Unable to Assess,Intubated or Tracheostomy,7
120642,A55687,I004,Verbal Response - Unable to Assess,Intubated or Tracheostomy,4


In [38]:
interventions_value_char = pd.read_sql_query('SELECT * FROM "lu.kuang".interventions_value_char',con=engine)
interventions_value_char

,ADMISSION_ID,ITEM_ID,LABEL,VALUE_CHAR,count
0,A00000,I311,Reason for Vent Monitoring INV,Blood Gas Analysis,4
1,A00000,I311,Reason for Vent Monitoring INV,Other (Add Comment),1
2,A00000,I311,Reason for Vent Monitoring INV,Parameter Change,4
3,A00000,I311,Reason for Vent Monitoring INV,Post Procedure,1
4,A00000,I311,Reason for Vent Monitoring INV,Scheduled Monitoring,5
...,...,...,...,...,...
255755,A55683,I331,Reason for Setting Change INV,Supports Daily Goals,2
255756,A55685,I331,Reason for Setting Change INV,Pt Condition Changed (Add Comment),2
255757,A55685,I331,Reason for Setting Change INV,Supports Daily Goals,1
255758,A55687,I331,Reason for Setting Change INV,By Protocol,1


In [39]:
# Select the columns from measurements
selected_measurements_vc = measurements_value_char.columns[4]
#selected_measurements_vc

In [40]:
selected_interventions_vc = interventions_value_char.columns[4]

In [41]:
pivoted_measurements_vc = measurements_value_char.pivot_table(index='ADMISSION_ID', columns=['ITEM_ID','VALUE_CHAR'], values=selected_measurements_vc)

In [42]:
pivoted_interventions_vc = interventions_value_char.pivot_table(index='ADMISSION_ID', columns=['ITEM_ID','VALUE_CHAR'], values=selected_interventions_vc)

In [43]:
#measurements value character
new_measurements_vc_columns = []
for column in pivoted_measurements_vc.columns:
    new_measurements_vc_column = f'{column[0]}_{column[1]}'
    new_measurements_vc_columns.append(new_measurements_vc_column)

In [44]:
#Intervetion value character
new_interventions_vc_columns = []
for column in pivoted_interventions_vc.columns:
    new_interventions_vc_column = f'{column[0]}_{column[1]}'
    new_interventions_vc_columns.append(new_interventions_vc_column)

In [48]:
# Assign the new column names
pivoted_measurements_vc.columns = new_measurements_vc_columns
combined_measurements_df = measurements_final_df.merge(pivoted_measurements_vc,on='ADMISSION_ID', how='outer')

In [49]:
pivoted_interventions_vc.columns = new_interventions_vc_columns
combined_interventions_df = interventions_final_df.merge(pivoted_interventions_vc,on='ADMISSION_ID', how='outer')

In [50]:
joined_df = master_df.merge(prescription_final_df, on='ADMISSION_ID', how='left')
joined_df = joined_df.merge(combined_interventions_df, on='ADMISSION_ID', how='left')
final_df = joined_df.merge(combined_measurements_df, on='ADMISSION_ID', how='left')

In [51]:
final_df

,PATIENT_ID,ADMISSION_ID,AGE_AT_ICU_ADMIT,GENDER,ICU_LOS,HOSP_LOS,ICU_EXPIRE_FLAG,HOSP_EXPIRE_FLAG,30_DAYS_EXPIRE_FLAG,DELIRIUM_FLAG,...,I019_Crushing,I019_Dull,I019_Heavy,I019_Intermittent,I019_Pressure,I019_Sharp,I019_Stabbing,I019_Throbbing,I019_With Coughing,I019_With Movement
0,P00000,A39362,71.0,F,61.9,62.8,1.0,1.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P00001,A25130,68.0,M,3.2,19.1,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P00002,A19016,53.0,M,3.8,14.1,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,P00004,A21002,68.0,M,2.5,2.5,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P00005,A15977,66.0,F,3.7,27.6,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,NaN,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47776,P48693,A55217,65.0,F,5.5,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47777,P48694,A55225,21.0,M,1.4,1.4,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47778,P48695,A55352,65.0,M,3.1,4.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47779,P48696,A55680,26.0,F,1.1,NaN,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [52]:
final_df.to_csv('final_df_317.csv', index=False)

## Impute Categorial Data

In [58]:
# Function to impute 0 for NaN cells if row sum is over 0 for columns with specific prefixes
def impute_zeros(row):
    relevant_columns = [col for col in row.index if any(col.startswith(prefix) for prefix in prefixes)]
    if row[relevant_columns].sum() > 0:
        return row.fillna(0)
    else:
        return row

In [59]:
# Selected Value Characters
prefixes = ['I311', 'I318', 'I323', 'I327', 'I331', 'I004', 'I016', 'I019']

In [60]:
final_df_1 = final_df.apply(impute_zeros, axis=1)
final_df_1

,PATIENT_ID,ADMISSION_ID,AGE_AT_ICU_ADMIT,GENDER,ICU_LOS,HOSP_LOS,ICU_EXPIRE_FLAG,HOSP_EXPIRE_FLAG,30_DAYS_EXPIRE_FLAG,DELIRIUM_FLAG,...,I019_Crushing,I019_Dull,I019_Heavy,I019_Intermittent,I019_Pressure,I019_Sharp,I019_Stabbing,I019_Throbbing,I019_With Coughing,I019_With Movement
0,P00000,A39362,71.0,F,61.9,62.8,1.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,P00001,A25130,68.0,M,3.2,19.1,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,P00002,A19016,53.0,M,3.8,14.1,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,P00004,A21002,68.0,M,2.5,2.5,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P00005,A15977,66.0,F,3.7,27.6,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47776,P48693,A55217,65.0,F,5.5,NaN,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47777,P48694,A55225,21.0,M,1.4,1.4,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
47778,P48695,A55352,65.0,M,3.1,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
47779,P48696,A55680,26.0,F,1.1,NaN,0.0,0.0,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
final_df_1.to_csv('final_df_317_1.csv', index=False)